In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "maintenance_iot")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("datalake", "adlsdatabricks1803test")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
datalake = dbutils.widgets.get("datalake")


ruta = f"abfss://{container}@{datalake}.dfs.core.windows.net/sensor_data.csv"

In [0]:
sensor_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("rig_id", StringType(), False),
    StructField("sensor_id", StringType(), False),
    StructField("sensor_type", StringType(), False),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

Paso 4 — Leer CSV


In [0]:
df_raw = spark.read \
    .option("header", True) \
    .schema(sensor_schema) \
    .csv(ruta)

print("Registros leídos:", df_raw.count())

In [0]:
from pyspark.sql.functions import current_timestamp, to_date, input_file_name

df_bronze = (
    df_raw
    .filter("value IS NOT NULL")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_date", to_date("event_timestamp"))
    .withColumn("source_file", input_file_name())
)

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ingestion_date") \
    .saveAsTable(f"{catalogo}.{esquema}.sensor_events")

In [0]:
%sql
SELECT COUNT(*) 
FROM maintenance_iot.bronze.sensor_events;

In [0]:
%sql
SELECT ingestion_date, COUNT(*) 
FROM maintenance_iot.bronze.sensor_events
GROUP BY ingestion_date;